In [ ]:
elec = pd.read_csv("data_raw/electricity/cost-of-electricity-by-country-2026.csv")
elec.columns


In [ ]:
[c for c in elec.columns if "202" in c]


In [ ]:
elec[elec["country"].isin(["United States", "Canada", "Australia", "Japan"])]


In [ ]:
import pandas as pd
import numpy as np

ember = pd.read_csv("data_raw/carbon/ember_yearly_electricity_data.csv")

ember["Area"].unique()[:50]


In [ ]:
[c for c in ["South Korea", "Singapore", "New Zealand"] if c in ember["Area"].unique()]


In [ ]:
countries_20 = [
    "Austria", "Belgium", "Denmark", "Finland", "France", "Germany", "Ireland",
    "Italy", "Netherlands", "Norway", "Poland", "Spain", "Sweden",
    "United States", "Canada", "Australia", "Japan",
    "South Korea", "Singapore", "New Zealand"
]


In [ ]:
elec = pd.read_csv("data_raw/electricity/cost-of-electricity-by-country-2026.csv")

year_cols = ["ElectricityCostUSDPerkWh_2022",
             "ElectricityCostUSDPerkWh_2023",
             "ElectricityCostUSDPerkWh_2024"]

elec_small = elec[elec["country"].isin(countries_20)].copy()
elec_small[year_cols] = elec_small[year_cols].apply(pd.to_numeric, errors="coerce")

elec_small["elec_price_usd_kwh"] = elec_small[year_cols].mean(axis=1)

elec_avg = elec_small[["country", "elec_price_usd_kwh"]]
elec_avg


In [ ]:
df_master = pd.DataFrame({"country": countries_20})
df_master = df_master.merge(elec_avg, on="country", how="left")

df_master[df_master["elec_price_usd_kwh"].isna()]


In [ ]:
ember = pd.read_csv("data_raw/carbon/ember_yearly_electricity_data.csv")
ember = ember[ember["Area"].isin(countries_20)]
ember = ember[ember["Year"].isin([2022, 2023, 2024])]

# CO2 intensity: already a rate, so average the yearly values directly.
carbon = ember[ember["Variable"] == "CO2 intensity"]
carbon_avg = (
    carbon.groupby("Area")["Value"]
    .mean()
    .reset_index()
    .rename(columns={"Area": "country", "Value": "carbon_intensity_gco2_kwh"})
)

# Renewable electricity share: calculate percentage share, not absolute generation.
renew_gen = ember[ember["Variable"] == "Renewables"][["Area", "Year", "Value"]].rename(
    columns={"Value": "renewable_generation_twh"}
)
total_gen = ember[ember["Variable"] == "Total Generation"][["Area", "Year", "Value"]].rename(
    columns={"Value": "total_generation_twh"}
)

renew_share = renew_gen.merge(total_gen, on=["Area", "Year"], how="inner")
renew_share = renew_share[renew_share["total_generation_twh"] > 0].copy()
renew_share["renew_share_pct"] = (
    renew_share["renewable_generation_twh"] / renew_share["total_generation_twh"] * 100
)

renew_avg = (
    renew_share.groupby("Area", as_index=False)
    .agg(
        renew_share_pct=("renew_share_pct", "mean"),
        renewable_generation_twh_2022_2024=("renewable_generation_twh", "mean"),
        total_generation_twh_2022_2024=("total_generation_twh", "mean"),
    )
    .rename(columns={"Area": "country"})
)

df_master = df_master.merge(carbon_avg, on="country", how="left")
df_master = df_master.merge(renew_avg, on="country", how="left")

df_master[df_master[["carbon_intensity_gco2_kwh", "renew_share_pct"]].isna().any(axis=1)]


In [ ]:
temp = pd.read_csv("data_raw/temperature/worldbank_avg_annual_temperature_2022_2024.csv")

year_cols = ["2022-07", "2023-07", "2024-07"]
temp_small = temp[temp["name"].isin(countries_20)].copy()

temp_small[year_cols] = temp_small[year_cols].apply(pd.to_numeric, errors="coerce")
temp_small["avg_temp_c"] = temp_small[year_cols].mean(axis=1)

temp_avg = temp_small[["name", "avg_temp_c"]].rename(columns={"name": "country"})
df_master = df_master.merge(temp_avg, on="country", how="left")


In [ ]:
df_master[df_master["avg_temp_c"].isna()]


In [ ]:
path = "data_raw/reliability/worldbank_transmission_losses.csv"
loss = pd.read_csv(path, skiprows=4)

loss_small = loss[loss["Country Name"].isin(countries_20)].copy()
year_cols = ["2022", "2023", "2024"]

loss_small[year_cols] = loss_small[year_cols].apply(pd.to_numeric, errors="coerce")
loss_small["t_d_losses_pct"] = loss_small[year_cols].mean(axis=1)

loss_avg = loss_small[["Country Name", "t_d_losses_pct"]].rename(columns={"Country Name": "country"})
df_master = df_master.merge(loss_avg, on="country", how="left")

df_master[df_master["t_d_losses_pct"].isna()]


In [ ]:
country_map = pd.DataFrame([
    # Europe
    ("Austria","AUT","AT"),
    ("Belgium","BEL","BE"),
    ("Denmark","DNK","DK"),
    ("Finland","FIN","FI"),
    ("France","FRA","FR"),
    ("Germany","DEU","DE"),
    ("Ireland","IRL","IE"),
    ("Italy","ITA","IT"),
    ("Netherlands","NLD","NL"),
    ("Norway","NOR","NO"),
    ("Poland","POL","PL"),
    ("Spain","ESP","ES"),
    ("Sweden","SWE","SE"),
    # Global
    ("United States","USA","US"),
    ("Canada","CAN","CA"),
    ("Australia","AUS","AU"),
    ("Japan","JPN","JP"),
    ("South Korea","KOR","KR"),
    ("Singapore","SGP","SG"),
    ("New Zealand","NZL","NZ"),
], columns=["country","iso3","iso2"])

country_map


In [ ]:
df_master = country_map[["country","iso3","iso2"]].copy()


In [ ]:
ember = pd.read_csv("data_raw/carbon/ember_yearly_electricity_data.csv")
ember = ember[ember["Year"].isin([2022, 2023, 2024])]
ember = ember[ember["ISO 3 code"].isin(country_map["iso3"])]
ember["Value"] = pd.to_numeric(ember["Value"], errors="coerce")

# CO2 intensity: already a rate, so average the yearly values directly.
carbon = ember[ember["Variable"] == "CO2 intensity"]
carbon_avg = (
    carbon.groupby("ISO 3 code")["Value"]
    .mean()
    .reset_index()
    .rename(columns={"ISO 3 code": "iso3", "Value": "carbon_intensity_gco2_kwh"})
)

# Renewable electricity share: calculate percentage share, not absolute generation.
# Ember's Variable == "Renewables" is an amount of generation/output, so using it
# directly would favour larger electricity systems. The model needs share (%).
renew_gen = ember[ember["Variable"] == "Renewables"][["ISO 3 code", "Year", "Value"]].rename(
    columns={"Value": "renewable_generation_twh"}
)
total_gen = ember[ember["Variable"] == "Total Generation"][["ISO 3 code", "Year", "Value"]].rename(
    columns={"Value": "total_generation_twh"}
)

renew_share = renew_gen.merge(total_gen, on=["ISO 3 code", "Year"], how="inner")
renew_share = renew_share[renew_share["total_generation_twh"] > 0].copy()
renew_share["renew_share_pct"] = (
    renew_share["renewable_generation_twh"] / renew_share["total_generation_twh"] * 100
)

renew_avg = (
    renew_share.groupby("ISO 3 code", as_index=False)
    .agg(
        renew_share_pct=("renew_share_pct", "mean"),
        renewable_generation_twh_2022_2024=("renewable_generation_twh", "mean"),
        total_generation_twh_2022_2024=("total_generation_twh", "mean"),
    )
    .rename(columns={"ISO 3 code": "iso3"})
)

df_master = df_master.merge(carbon_avg, on="iso3", how="left")
df_master = df_master.merge(renew_avg, on="iso3", how="left")

df_master[df_master[["carbon_intensity_gco2_kwh", "renew_share_pct"]].isna().any(axis=1)]


In [ ]:
temp = pd.read_csv("data_raw/temperature/worldbank_avg_annual_temperature_2022_2024.csv")

temp_year_cols = ["2022-07", "2023-07", "2024-07"]

temp_small = temp[temp["code"].isin(country_map["iso3"])].copy()
temp_small[temp_year_cols] = temp_small[temp_year_cols].apply(pd.to_numeric, errors="coerce")
temp_small["avg_temp_c"] = temp_small[temp_year_cols].mean(axis=1)

temp_avg = temp_small[["code","avg_temp_c"]].rename(columns={"code":"iso3"})

df_master = df_master.merge(temp_avg, on="iso3", how="left")

df_master[df_master["avg_temp_c"].isna()][["country","iso3"]]


In [ ]:
loss = pd.read_csv("data_raw/reliability/worldbank_transmission_losses.csv", skiprows=4)

loss_small = loss[loss["Country Code"].isin(country_map["iso3"])].copy()
year_cols = ["2022","2023","2024"]

loss_small[year_cols] = loss_small[year_cols].apply(pd.to_numeric, errors="coerce")
loss_small["t_d_losses_pct"] = loss_small[year_cols].mean(axis=1)

loss_avg = loss_small[["Country Code","t_d_losses_pct"]].rename(columns={"Country Code":"iso3"})

df_master = df_master.merge(loss_avg, on="iso3", how="left")

df_master[df_master["t_d_losses_pct"].isna()][["country","iso3"]]


In [ ]:
elec = pd.read_csv("data_raw/electricity/cost-of-electricity-by-country-2026.csv")

year_cols = ["ElectricityCostUSDPerkWh_2022",
             "ElectricityCostUSDPerkWh_2023",
             "ElectricityCostUSDPerkWh_2024"]

elec_small = elec[elec["flagCode"].isin(country_map["iso2"])].copy()
elec_small[year_cols] = elec_small[year_cols].apply(pd.to_numeric, errors="coerce")
elec_small["elec_price_usd_kwh"] = elec_small[year_cols].mean(axis=1)

elec_avg = elec_small[["flagCode","elec_price_usd_kwh"]].rename(columns={"flagCode":"iso2"})

df_master = df_master.merge(elec_avg, on="iso2", how="left")

df_master[df_master["elec_price_usd_kwh"].isna()][["country","iso2"]]


In [ ]:
df_master.isna().sum()
df_master


In [ ]:
df_master.to_csv("data_processed/country_indicators_global_2022_2024.csv", index=False)


In [ ]:
# Data quality check: renewable share should be a percentage between 0 and 100.
assert df_master["renew_share_pct"].between(0, 100).all(), (
    "renew_share_pct contains values outside 0–100. Check the Ember renewable share calculation."
)

df_master[["country", "renew_share_pct", "renewable_generation_twh_2022_2024", "total_generation_twh_2022_2024"]]     .sort_values("renew_share_pct", ascending=False)     .head(10)


In [ ]:
df = df_master.copy()

lower_better = ["elec_price_usd_kwh", "carbon_intensity_gco2_kwh", "avg_temp_c", "t_d_losses_pct"]
higher_better = ["renew_share_pct"]

def minmax(s):
    return (s - s.min()) / (s.max() - s.min())

for c in lower_better:
    df[c + "_norm"] = 1 - minmax(df[c])

for c in higher_better:
    df[c + "_norm"] = minmax(df[c])

norm_cols = [c + "_norm" for c in lower_better + higher_better]
df[["country"] + norm_cols].head()


In [ ]:
import numpy as np

X = df[norm_cols].copy()
eps = 1e-12

P = X / (X.sum(axis=0) + eps)
n = len(X)
k = 1 / np.log(n)

entropy = -k * (P * np.log(P + eps)).sum(axis=0)
d = 1 - entropy
w_entropy = (d / d.sum()).sort_values(ascending=False)

w_entropy


In [ ]:
df["entropy_score"] = (df[w_entropy.index] * w_entropy.values).sum(axis=1)
df[["country", "entropy_score"]].sort_values("entropy_score", ascending=False).head(10)


In [ ]:
import numpy as np

def run_sensitivity(df, base_weights, n_sim=1000, perturb=0.15):
    
    cols = list(base_weights.keys())
    base = np.array(list(base_weights.values()))
    
    top3_counts = {country: 0 for country in df["country"]}
    
    for _ in range(n_sim):
        
        # perturb weights
        noise = np.random.uniform(1-perturb, 1+perturb, size=len(base))
        w = base * noise
        w = w / w.sum()
        
        scores = (df[cols] * w).sum(axis=1)
        ranked = df.assign(score=scores).sort_values("score", ascending=False)
        
        top3 = ranked.head(3)["country"].tolist()
        
        for c in top3:
            top3_counts[c] += 1
    
    sensitivity_df = pd.DataFrame({
        "country": list(top3_counts.keys()),
        "top3_frequency": list(top3_counts.values())
    })
    
    sensitivity_df["top3_percent"] = 100 * sensitivity_df["top3_frequency"] / n_sim
    
    return sensitivity_df.sort_values("top3_percent", ascending=False)


In [ ]:
balanced_weights = {c: 1/len(norm_cols) for c in norm_cols}

sens_balanced = run_sensitivity(df, balanced_weights, n_sim=2000, perturb=0.15)
sens_balanced.head(10)


In [ ]:
sens_balanced_40 = run_sensitivity(df, balanced_weights, n_sim=2000, perturb=0.40)
sens_balanced_40.head(10)


In [ ]:
sustain_weights = {
    "elec_price_usd_kwh_norm": 0.15,
    "carbon_intensity_gco2_kwh_norm": 0.35,
    "renew_share_pct_norm": 0.30,
    "avg_temp_c_norm": 0.10,
    "t_d_losses_pct_norm": 0.10
}

sens_sustain = run_sensitivity(df, sustain_weights, n_sim=2000, perturb=0.15)
sens_sustain.head(10)


In [ ]:
df[[
    "country",
    "carbon_intensity_gco2_kwh_norm",
    "renew_share_pct_norm"
]].sort_values("carbon_intensity_gco2_kwh_norm", ascending=False).head(10)


In [ ]:
df[[
    "country",
    "carbon_intensity_gco2_kwh_norm",
    "renew_share_pct_norm"
]].sort_values("renew_share_pct_norm", ascending=False).head(10)


In [ ]:
df_master[["country", "renew_share_pct"]].sort_values("renew_share_pct", ascending=False).head(10)


In [ ]:
balanced_scores = (df[norm_cols] * (1/len(norm_cols))).sum(axis=1)
df["balanced_score"] = balanced_scores

df["balanced_rank"] = df["balanced_score"].rank(ascending=False)
df["entropy_rank"] = df["entropy_score"].rank(ascending=False)

df[["country","balanced_rank","entropy_rank"]].sort_values("balanced_rank")


In [ ]:
df[["balanced_rank","entropy_rank"]].corr(method="spearman")


In [ ]:
def custom_score(df, weights):
    w = pd.Series(weights)
    return (df[w.index] * w).sum(axis=1)


In [ ]:
import pandas as pd

df_main = pd.read_csv("data_processed/country_indicators_global_2022_2024.csv")
df_dc = pd.read_csv("data_processed/datacentre_facility_counts.csv")

merge_cols = ["country"]
if "iso3" in df_main.columns and "iso3" in df_dc.columns:
    merge_cols = ["country", "iso3"]

df_main = df_main.merge(
    df_dc[merge_cols + ["dc_facility_count", "dc_facility_count_log", "dc_maturity_norm"]],
    on=merge_cols,
    how="left"
)

df_main.head()

In [ ]:
df_main[["country", "iso3", "dc_facility_count"]].sort_values("country")

In [ ]:
df_main[df_main["dc_facility_count"].isna()][["country", "iso3"]]

In [ ]:
import pandas as pd

df_dc = pd.read_csv("data_processed/datacentre_facility_counts.csv")
df_dc.head()

In [ ]:
dc_rank = df_dc[["country", "iso3", "dc_facility_count", "dc_facility_count_log"]].copy()
dc_rank = dc_rank.sort_values("dc_facility_count", ascending=False).reset_index(drop=True)
dc_rank["dc_rank"] = dc_rank.index + 1

dc_rank

In [ ]:
dc_rank.to_csv("outputs/datacentre_facility_rankings.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt

plot_df = dc_rank.sort_values("dc_facility_count", ascending=False).head(20)

plt.figure(figsize=(10, 6))
plt.bar(plot_df["country"], plot_df["dc_facility_count"])
plt.xticks(rotation=45, ha="right")
plt.xlabel("Country")
plt.ylabel("Scraped data centre facility count")
plt.title("Scraped data centre facility counts by country")
plt.tight_layout()
plt.savefig("outputs/fig_scraped_datacentre_counts.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
top_dc = dc_rank.head(5)
bottom_dc = dc_rank.tail(5)

print("Top 5 countries by scraped facility count:")
print(top_dc.to_string(index=False))

print("\nBottom 5 countries by scraped facility count:")
print(bottom_dc.to_string(index=False))